In [ ]:
!pip install scikit-learn

In [14]:
import math
import numpy
import pandas as pd
from sklearn.model_selection import train_test_split


In [ ]:
spam_ham_dataset = pd.read_csv("spam_ham_dataset.csv")

In [11]:
spam_ham_dataset.head()
X = spam_ham_dataset.drop(columns = ["Unnamed: 0","label", "label_num"])
y = spam_ham_dataset["label_num"]

In [15]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2,       # 20% allocated to test set
    random_state=42,     # Ensures reproducible splits
    stratify=y           # Keeps target class proportions identical (optional)
)

In [29]:
print(X_train.head())


                                                   text
656   Subject: re : the hstoett lady sucklng huge cc...
4752  Subject: lose 19 % , powerful weightloss now a...
4483  Subject: 98 - 2601\r\nhi daren ,\r\ni ' m atte...
1109  Subject: 2 nd rev mar . 2000 josey ranch nom\r...
711   Subject: lose it\r\nos effetiveeight os aaiabe...


In [26]:
print(y_train.head())
print(len(y_train))
print("spam = ", (y_train==1).sum())
print("ham = ", (y_train==0).sum())

656     1
4752    1
4483    0
1109    0
711     1
Name: label_num, dtype: int64
4136
spam =  1199
ham =  2937


In [111]:
log_prior_ham = math.log((y_train == 0).sum()/len(y_train))
log_prior_spam = math.log((y_train == 1).sum()/len(y_train))
print(log_prior_ham,log_prior_spam)

-0.342340484989645 -1.238241261160751


In [117]:
total_vocab = set(
    X_train["text"]
    .str.findall(r"\b[a-zA-Z]+\b")
    .explode()
    .str.lower()
    .dropna()
    .unique()
    .tolist()
    )

In [37]:
print(len(total_vocab))
print(total_vocab[:100])

43824
['subject', 're', 'the', 'hstoett', 'lady', 'sucklng', 'huge', 'ccok', 'what', 's', 'your', 'pleasure', 'squire', 'kuwo', 'lose', '19', 'powerful', 'weightloss', 'now', 'available', 'where', 'you', 'are', 'hello', 'i', 'have', 'a', 'special', 'offer', 'for', 'want', 'to', 'weight', 'most', 'is', 'without', 'prescription', 'all', 'natural', 'adipren', '720', '100', 'money', 'back', 'guarant', 'e', 'up', 'total', 'body', 'loss', 'of', '20', '35', 'abdominal', 'fat', '300', 'more', 'while', 'dieting', 'increase', 'metabolic', 'rate', 'by', '76', '9', 'exercise', 'reduction', '40', '70', 'overall', 'under', 'skin', 'suppresses', 'appetite', 'sugar', 'burns', 'calorized', 'boost', 'confidence', 'level', 'and', 'self', 'esteem', 'get', 'facts', 'about', 'http', 'www', '1', 'com', 'system', 'information', 'architecture', 'identical', 'example', 'locale', 'software', 'properties', 'specific', 'entities']


In [ ]:
ham_indices = y_train[y_train == 0].index
spam_indices = y_train[y_train == 1].index

In [52]:
ham_indices[0]

np.int64(4483)

In [113]:
total_words_ham = int(
    X_train.loc[ham_indices,"text"]
    .str.findall(r"\b[a-zA-Z]+\b")
    .explode()
    .str.lower()
    .dropna()
    .count()
) + len(total_vocab)
total_words_spam = int(
    X_train.loc[spam_indices,"text"]
    .str.findall(r"\b[a-zA-Z]+\b")
    .explode()
    .str.lower()
    .dropna()
    .count()
)+len(total_vocab)

In [65]:
ham_word_frequencies = (
    X_train.loc[ham_indices,"text"]
    .str.findall(r"\b[a-zA-Z]+\b")
    .explode()
    .str.lower()
    .dropna()
    .value_counts()
)

spam_word_frequencies = (
    X_train.loc[spam_indices,"text"]
    .str.findall(r"\b[a-zA-Z]+\b")
    .explode()
    .str.lower()
    .dropna()
    .value_counts()
)

In [115]:
ham_word_frequencies_dict = ham_word_frequencies.to_dict()
spam_word_frequencies_dict = spam_word_frequencies.to_dict()

In [121]:
import re
import numpy as np
word_match_regex = r"\b([a-zA-Z]+)\b"
total_test_samples = len(y_test)
correct_count = 0
def get_ham_word_log_prob(word):
    word = str.lower(word)
    count = ham_word_frequencies_dict.get(word,0)
    return np.log((count+1)/total_words_ham)
def get_spam_word_log_prob(word):
    word = str.lower(word)
    count = spam_word_frequencies_dict.get(word,0)
    return np.log((count+1)/total_words_spam)

for row in X_test.itertuples():
    true_label = y_test.loc[row[0]]
    email = row[1]
    nwords = re.findall(word_match_regex,email)
    words = [str.lower(w) for w in nwords if str.lower(w) in total_vocab]
    hamscore =  sum([get_ham_word_log_prob(word) for word in words]) + log_prior_ham
    spamscore = sum([get_spam_word_log_prob(word) for word in words]) + log_prior_spam
    predicted = int(spamscore>hamscore)
    if predicted==true_label:
        correct_count+=1
    print(f"true label is {true_label}, prediction is {predicted}")

print("Accuracy = ", (correct_count/total_test_samples))

    

true label is 0, prediction is 0
true label is 0, prediction is 0
true label is 0, prediction is 0
true label is 0, prediction is 0
true label is 0, prediction is 0
true label is 1, prediction is 1
true label is 0, prediction is 0
true label is 0, prediction is 0
true label is 0, prediction is 0
true label is 1, prediction is 1
true label is 0, prediction is 0
true label is 0, prediction is 0
true label is 1, prediction is 1
true label is 1, prediction is 1
true label is 0, prediction is 0
true label is 0, prediction is 0
true label is 1, prediction is 1
true label is 0, prediction is 0
true label is 0, prediction is 0
true label is 0, prediction is 0
true label is 1, prediction is 1
true label is 1, prediction is 1
true label is 1, prediction is 1
true label is 0, prediction is 0
true label is 0, prediction is 0
true label is 0, prediction is 0
true label is 0, prediction is 0
true label is 0, prediction is 0
true label is 1, prediction is 1
true label is 1, prediction is 1
true label